In [0]:
dbutils.widgets.text("catalog", "weather_dev", "Catálogo de destino")
catalog = dbutils.widgets.get("catalog")

In [0]:
import requests
import json

latitude = 19.4326
longitude = -99.1332

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m,precipitation,wind_speed_10m",
    "timezone": "America/Mexico_City"
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

print(json.dumps(data, indent=2)[:1000])

In [0]:
from datetime import datetime, timezone

ingestion_ts = datetime.now(timezone.utc)
ingestion_date = ingestion_ts.date().isoformat()

bronze_row = {
    "ingestion_timestamp": ingestion_ts,
    "ingestion_date": ingestion_date,
    "source": "open-meteo",
    "latitude": latitude,
    "longitude": longitude,
    "raw_response": json.dumps(data)
}

df_bronze = spark.createDataFrame([bronze_row])
df_bronze.printSchema()
display(df_bronze)

In [0]:
table_name = f"{catalog}.bronze.weather_forecast_raw"

(df_bronze.write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .saveAsTable(table_name)
)

display(spark.sql(f"SELECT * FROM {table_name}"))